In [39]:
%pip install -q transformers

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import math
import transformers

device = 'cuda' if torch.cuda.is_available() else 'cpu'

batch_size, seq_len, emb_dim = 2, 5, 32
head_dim = 8

x = torch.randn(batch_size, seq_len, emb_dim).to(device)
padding_mask = torch.ones(seq_len, seq_len)
padding_mask[0,0] = 0

In [16]:
class MultiheadSelfAttn(torch.nn.Module):
  def __init__(self, emb_dim=32, num_heads=4):
    super().__init__()
    self.head_dim = emb_dim // num_heads
    self.num_heads = num_heads

    self.attn = torch.nn.Linear(emb_dim, emb_dim * 3)
    self.output_proj = torch.nn.Linear(emb_dim, emb_dim)

  # x: (batch_size, seq_len, emb_dim)
  def forward(self, x):
    batch_size, seq_len, emb_dim = x.size(0), x.size(1), x.size(2)
    
    x = self.attn(x)
    q, k, v = torch.split(x, emb_dim, dim=-1)
    
    q = q.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    k = k.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    v = v.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    
    print('q after reshape', q.size())
    
    attn_weights = (q @ k.transpose(-1, -2)) / math.sqrt(self.head_dim)
    print('attn_weights', attn_weights.size())
    
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool).tril(diagonal=0)
    attn_weights = torch.where(mask == True, attn_weights, float('-inf'))
    
    attn_weights = torch.nn.functional.softmax(attn_weights, dim=-1)
    
    attn_result = attn_weights @ v
    
    attn_result = attn_result.permute(0, 2, 1, 3).reshape(batch_size, seq_len, emb_dim)
    
    return self.output_proj(attn_result)
  

MultiheadSelfAttn(emb_dim=emb_dim).to(device)(x).size()

q after reshape torch.Size([2, 4, 5, 8])
attn_weights torch.Size([2, 4, 5, 5])


torch.Size([2, 5, 32])

In [3]:
tokenizer = transformers.AutoTokenizer.from_pretrained('bert-base-uncased')

KeyboardInterrupt: 

In [ ]:
lines = [
    "Luke, I am your father.",
    "Life is what happens when you're busy making other plans.",
    ]

tokens_info = tokenizer(lines, padding=True, truncation=True, return_tensors="pt")

print(tokens_info['input_ids'])
# tensor([[ 101, 5355, 1010, 1045, 2572, 2115, 2269, 1012,  102,    0,    0,    0, 0,    0,    0],
#        [ 101, 2166, 2003, 2054, 6433, 2043, 2017, 1005, 2128, 5697, 2437, 2060, 3488, 1012,  102]])

padding_mask = tokens_info['input_ids'] != 0 # torch.Size([2, 15])

KeyboardInterrupt: 

In [ ]:
class SelfAttn(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = torch.nn.Linear(32, 32)
        self.k_proj = torch.nn.Linear(32, 32)
        self.v_proj = torch.nn.Linear(32, 32)
        
    # x torch.Size([2, 15, 32])
    def forward(self, x, key_padding_mask):
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        
        attn_weights = q @ torch.transpose(k, -2, -1) * (1 / math.sqrt(k.size(-1)))
        
        attn_weights = torch.where(padding_mask == 1, attn_weights, float('-inf'))
        
        if key_padding_mask is not None:
            # Expand mask to (batch, 1, seq_len) and invert: True means "mask this position"
            mask = key_padding_mask[:, None, :] == 0   # True for padding keys
            attn_weights = attn_weights.masked_fill(mask, float('-inf'))
        
        attn_result = torch.nn.functional.softmax(attn_weights, dim=-1) @ v
        
        print(attn_result)
        
        return attn_result
        
SelfAttn().to(device)(torch.randn(2, 15, 32), key_padding_mask=tokens_info['attention_mask']).size()

RuntimeError: The size of tensor a (2) must match the size of tensor b (15) at non-singleton dimension 1